### Random vs Random

In [9]:
import time
from collections import Counter, defaultdict
from domain.configs import MAX_STEPS_PER_EPISODE, EVALUATE_GAMES
from environment.grenight_environment import GrenightEnvironment

In [10]:
def play_random_game(env_arg: GrenightEnvironment) -> tuple[str, dict, int]:

    env_arg.reset()
    done = False
    move_count = 0
    acting_player_is_white = True
    reward = 0.0
    info = None

    while not done and move_count < MAX_STEPS_PER_EPISODE:
        acting_player_is_white = env_arg.is_white_on_turn
        action = env_arg.sample()
        _, reward, done, info = env_arg.step(action)
        move_count += 1

    if not done:
        return "truncated", info, move_count
    if reward == 0.0:
        return "draw", info, move_count

    winner_is_white = acting_player_is_white if reward == 1.0 else not acting_player_is_white

    return ("white_win", info, move_count) if winner_is_white else ("black_win", info, move_count)

In [20]:
env = GrenightEnvironment()
outcomes_counter = Counter()
draw_reasons_counter = Counter()
total_moves_per_outcome = defaultdict(int)

start_time = time.perf_counter()
for _ in range(EVALUATE_GAMES):
    outcome, game_info, move_count = play_random_game(env)
    outcomes_counter[outcome] += 1
    total_moves_per_outcome[outcome] += move_count
    if game_info["draw_reason"] is not None:
        draw_reasons_counter[game_info["draw_reason"]] += 1
end_time = time.perf_counter()

average_moves_per_outcome = dict()
for outcome, total_moves in total_moves_per_outcome.items():
    average_moves_per_outcome[outcome] = f"{(total_moves / outcomes_counter[outcome]):.2f}"

print(f"STATS OUT FROM: {EVALUATE_GAMES} GAMES\n"
      f"Outcomes: {outcomes_counter}\n"
      f"Average moves per outcome: {average_moves_per_outcome}\n"
      f"Total average moves: {sum(total_moves_per_outcome.values()) / EVALUATE_GAMES}\n"
      f"Draw reasons: {draw_reasons_counter}\n"
      f"Execution time: {end_time - start_time:.2f} seconds\n")

STATS OUT FROM: 625 GAMES
Outcomes: Counter({'draw': 433, 'black_win': 99, 'white_win': 93})
Average moves per outcome: {'draw': '78.25', 'white_win': '22.48', 'black_win': '24.87'}
Total average moves: 61.496
Draw reasons: Counter({'insufficient_material': 249, 'stalemate': 109, 'max_steps_without_progress': 56, 'threefold_repetition': 19})
Execution time: 38.69 seconds

